In [ ]:
from snowflake.snowpark.context import get_active_session
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Active Snowflake session
session = get_active_session()

# Table coordinates
DATABASE_NAME = 'BIKE_TRIP_ANALYSIS'
SCHEMA_NAME = 'PUBLIC'
TABLE_NAME = 'month_data'

full_table_path = f"{DATABASE_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
print(f"Connected to session. Target Table: {full_table_path}")

In [ ]:
# Fetch from Snowflake and convert directly to a Pandas DataFrame
df = session.table(full_table_path).to_pandas()

# Normalize column names to lowercase to match your original script
df.columns = df.columns.str.lower().str.strip()

print("Combined shape: ", df.shape)
df.head()

In [ ]:
# replacing missing station names.

df['start_station_name'] = df['start_station_name'].fillna('On-Street / Dockless Lock')
df['end_station_name'] = df['end_station_name'].fillna('On-Street / Dockless Lock')
missing_summary = pd.DataFrame({
    'Column Name' : df.columns,
    'Missing Count'# convert timestamp columns into datetime format

df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# drop duplicate ride records if any

df.drop_duplicates(subset=['ride_id'], inplace = True)

# making column name into readable

df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns) : df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary

In [ ]:
# convert timestamp columns into datetime format

df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# drop duplicate ride records if any

df.drop_duplicates(subset=['ride_id'], inplace = True)

In [ ]:
# replacing missing station names.

df['start_station_name'] = df['start_station_name'].fillna('On-Street / Dockless Lock')
df['end_station_name'] = df['end_station_name'].fillna('On-Street / Dockless Lock')

In [ ]:
# feature engineering.

# 1. Ride duration in minutes.

df['ride_length_mins'] = ((df['ended_at'] - df['started_at']).dt.total_seconds() / 60.0 ).round(2)

# 2 Outlier Filtering: Remove bounce trips (<1 min) and abandoned rides (>24 hrs / 1440 mins)
# Also filter coordinate records within the Greater Chicago Metro bounds

valid_duration = df['ride_length_mins'].between(1.0, 1440.0)
valid_coords = (
    df['start_lat'].between(41.5, 42.2) &
    df['start_lng'].between(-88.0, -87.4) &
    df['end_lat'].between(41.5, 42.2) &
    df['end_lng'].between(-88.0, -87.4)
)

clean_df = df[valid_duration & valid_coords].copy()
print(f"Cleaned Records: {len(clean_df):,} ({len(clean_df)/len(df)*100:.2f}% retained)")

# 2. Round Trip / Leisure Loop Flag

clean_df['is_round_trip'] = (
    (clean_df['start_station_name'] == clean_df['end_station_name']) &
    (clean_df['start_station_name'] != 'On-Street / Dockless Lock')
).astype(int)

# 3. Temporal Attributes

clean_df['month_name'] = clean_df['started_at'].dt.strftime('%B')
clean_df['day_name'] = clean_df['started_at'].dt.day_name()
clean_df['day_of_week'] = clean_df['started_at'].dt.dayofweek # 0=Mon, 6=Sun
clean_df['hour'] = clean_df['started_at'].dt.hour
clean_df['is_weekend'] = clean_df['day_of_week'].isin([5, 6]).astype(int)

# 4. Rush Hour / Commuter Flag (Mon-Fri 7-9 AM & 4-6 PM)

clean_df['is_commute_window'] = (
    (clean_df['is_weekend'] == 0) &
    (clean_df['hour'].isin([7, 8, 9, 16, 17, 18]))
).astype(int)

# 5. Trip Duration Categories (qcut / binning)

duration_bins = [0, 10, 20, 45, 1440]
duration_labels = ['Short (<10m)', 'Medium (10-20m)', 'Long (20-45m)', 'Extended (>45m)']
clean_df['duration_category'] = pd.cut(clean_df['ride_length_mins'], bins=duration_bins, labels=duration_labels)

In [ ]:
# EDA

# 1. Diurnal Hourly Trend: Commuter Peaks vs. Casual Curve
plt.figure(figsize=(12, 5))
hourly_demand = clean_df.groupby(['hour', 'member_casual'])['ride_id'].count().unstack()
hourly_demand.plot(kind='line', marker='o', ax=plt.gca(), color=['#2969B0', '#0E2B52'])
plt.title('Hourly Trip Demand: Member (Bi-Modal Commute Peaks) vs. Casual (Mid-Day Leisure)', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Total Trips')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Day-of-Week Distribution
plt.figure(figsize=(10, 5))
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_trends = clean_df.groupby(['day_name', 'member_casual'])['ride_id'].count().unstack().loc[days_order]
day_trends.plot(kind='bar', ax=plt.gca(), color=['#2969B0', '#0E2B52'])
plt.title('Trip Volume by Day of Week: Weekday Utility vs. Weekend Surge', fontsize=13, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Total Rides')
plt.xticks(rotation=45)
plt.legend(title='User Category')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Top 10 Transit Corridors
station_routes = clean_df[
    (clean_df['start_station_name'] != 'On-Street / Dockless Lock') &
    (clean_df['end_station_name'] != 'On-Street / Dockless Lock')
].groupby(['start_station_name', 'end_station_name']).size().reset_index(name='trip_count')

top_10 = station_routes.sort_values(by='trip_count', ascending=False).head(10)
top_10['route'] = top_10['start_station_name'] + ' ➔ ' + top_10['end_station_name']

plt.figure(figsize=(12, 6))
sns.barplot(data=top_10, y='route', x='trip_count', palette='Blues_r')
plt.title('Top 10 High-Traffic Station-to-Station Transit Corridors', fontsize=13, fontweight='bold')
plt.xlabel('Total Rides')
plt.ylabel('Transit Route')
plt.tight_layout()
plt.show()

In [ ]:
# Reset index to a standard RangeIndex before writing
clean_df = clean_df.reset_index(drop=True)

# Write to Snowflake without warnings
session.write_pandas(
    df=clean_df,
    table_name="CLEAN_MONTH_DATA",
    database=DATABASE_NAME,
    schema=SCHEMA_NAME,
    auto_create_table=True,
    overwrite=True
)
print("Clean table successfully saved to Snowflake!")